<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom:3px solid rgb(255,106,0); padding-bottom:1em;'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 5: Full Training Loop</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

<!-- FULL: keep, DEMO: delete -->
*All pieces in place. Next: assemble them into a complete training loop.*

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt
import sys
sys.path.append('../../A2/')
from helpers import load_data

X, y = load_data('../../A2/data.csv')
from plotting import plot_losses

class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

dataset = TensorDataset(X, y)
torch.manual_seed(0)
n_train = int(0.8 * len(dataset))
train_dataset, val_dataset = random_split(dataset, [n_train, len(dataset)-n_train])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)


<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Training and validation functions
</div>

<!-- FULL: keep, DEMO: delete -->
Separate training and validation logic into reusable functions.
Note `model.train()` / `model.eval()` and `torch.no_grad()` in the validation loop.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """One epoch of training. Returns mean loss."""
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


def val_epoch(model, loader, loss_fn):
    """One epoch of validation. Returns mean loss."""
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred     = model(X_batch)
            loss       = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)


print('Functions defined.')

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(179,27,27); background:rgb(255,240,240); color:rgb(179,27,27); font-size:1.1em; font-weight:bold;'>
  &#9888; Failure case &mdash; forgetting model.eval() and torch.no_grad() in validation
</div>

<!-- FULL: keep, DEMO: delete -->
Without `model.eval()`, layers like Dropout and BatchNorm (Session 7) behave as if training — validation metrics will be wrong.

Without `torch.no_grad()`, PyTorch builds and retains the computation graph for every validation batch — memory grows unnecessarily and validation is slower.

In [ ]:
# Wrong — no eval(), no no_grad()
def val_epoch_broken(model, loader, loss_fn):
    total_loss = 0.0
    for X_batch, y_batch in loader:
        y_pred     = model(X_batch)          # graph built unnecessarily
        loss       = loss_fn(y_pred, y_batch)
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

# Correct — always use both
def val_epoch_correct(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred     = model(X_batch)
            loss       = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

<!-- FULL: keep, DEMO: shorten -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Training MLP on housing data
</div>

In [ ]:
torch.manual_seed(0)
model     = MLP(in_features=4, hidden=16, out_features=1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

n_epochs    = 100
train_losses = []
val_losses   = []

for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    val_loss   = val_epoch(model, val_loader, loss_fn)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}  train loss: {train_loss:.4f}  val loss: {val_loss:.4f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, color='#005564', label='train loss')
ax.plot(val_losses,   color='#FF6A00', label='val loss')
ax.set_xlabel('epoch')
ax.set_ylabel('MSE loss')
ax.set_title('Training curve')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# cleaner plotting using helpers
from plotting import plot_losses
plot_losses(train_losses, val_losses)

<!-- FULL: keep, DEMO: delete -->
<div style='margin:2em 0 0.5em 0; padding:0.4em 0.8em; border-left:5px solid rgb(255,106,0); background:rgb(240,244,248); color:rgb(22,60,105); font-size:1.1em; font-weight:bold;'>
  Reading loss curves
</div>

<!-- FULL: keep, DEMO: delete -->
- **Both losses decrease** — training is working
- **Train loss $\ll$ val loss** — overfitting; model memorises training data
- **Val loss stops improving** — stop training or regularise
- **Loss spikes suddenly** — learning rate too large

Overfitting and regularisation covered in Session 7.

<!-- FULL: keep, DEMO: keep -->
<br/>